In [10]:
import pandas as pd

df = pd.read_csv("datatelco_customer_churn.csv")

pd.to_numeric(df['TotalCharges'], errors='coerce')
df.head()
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

# El problema con las columnas que no son reconocidas directamente como numéricas

Luego de investigar un poco, descubrimos que Pandas infiere el dtype al leer el CSV, columna por columna, y aplica una regla simple: si todos los valores de la columna se pueden convertir a número, la hace numérica pero si aparece aunque sea uno solo que no puede, toda la columna cae a object.Object significa que la columna no guarda los valores en sí, sino punteros a objetos de Python. En la práctica, casi siempre son strings.

El caso que encontramos fue con TotalCharges. Hay 7043 filas, de las cuales 7032 son montos perfectamente numéricos. Pero 11 tienen un espacio en blanco. Ese espacio no se puede convertir a número, así que pandas se rinde y guarda las 7043 como strings. Un "29.85" con comillas, no un 29.85.

Los espacios en blanco aparecen cuando son clientes con tenure = 0, o sea que recién se dieron de alta y todavía no facturaron nada. Esa columna sí es numérica conceptualmente, pero pandas la lee como object porque hay unas 11 filas con un string vacío (" ") en lugar de un número.

El impacto de no detectarlo es que la columna queda como texto y el modelo o la ignora, o si alguien la codifica como categórica, termina tratando cada monto como una categoría distinta.

Se arregla con pd.to_numeric(df['TotalCharges'], errors='coerce') y después tendríamos que decidir qué hacer con esos NaN (imputar con 0 tiene sentido acá, justificando que no facturaron todavía).

In [11]:
# 1. Conversión: los " " no parseables pasan a NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 2. Verificación del diagnóstico (para dejar registro en el notebook)
faltantes = df['TotalCharges'].isna()
print(f"Filas con TotalCharges nulo: {faltantes.sum()}")
print(f"Valores de tenure en esas filas: {df.loc[faltantes, 'tenure'].unique()}")

# 3. Imputación
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# 4. Chequeo final
print(f"\nDtype: {df['TotalCharges'].dtype}")
print(f"Nulos restantes: {df['TotalCharges'].isna().sum()}")
print(f"Mínimo: {df['TotalCharges'].min()}")

df.dtypes

Filas con TotalCharges nulo: 11
Valores de tenure en esas filas: [0]

Dtype: float64
Nulos restantes: 0
Mínimo: 0.0


customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object